# Étape 2 — Embeddings et indexation FAISS

**Objectif de ce notebook :**
- Reprendre les chunks générés à l'étape 1
- Les transformer en vecteurs numériques (embeddings) avec un modèle multilingue
- Construire un index FAISS (structure permettant de retrouver rapidement les chunks
  les plus proches d'une question)
- Tester une recherche de similarité "à la main", sans LLM, pour bien comprendre ce que
  fait le retriever avant de brancher Ollama à l'étape 3
- Sauvegarder l'index sur disque pour le réutiliser dans l'API

**Kernel à utiliser :** `Python (RAG GED)`.

## 0. Imports et rechargement des chunks (étape 1)

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
sys.path.append(str(SRC_DIR))

from ingest import load_and_split, DATA_DIR

chunks = load_and_split(DATA_DIR)
print(f"{len(chunks)} chunks rechargés depuis data/docs/.")

[ingest] Chargé : exemple_atelier_preparation.md
[ingest] Chargé : GDS_GED_Services (4).pdf
63 chunks rechargés depuis data/docs/.


## 2a. Choix et chargement du modèle d'embeddings

On utilise un modèle **multilingue** (`paraphrase-multilingual-MiniLM-L12-v2`), important
car ta documentation est en français. Il transforme n'importe quel texte en un vecteur
de 384 nombres (dimensions) qui capture son "sens".

⚠️ Le premier chargement télécharge le modèle depuis Hugging Face (~470 Mo) — ça peut
prendre quelques minutes selon ta connexion. Les fois suivantes seront quasi instantanées
(le modèle reste en cache localement).

In [2]:
from build_index import get_embeddings, EMBEDDING_MODEL_NAME

print("Chargement du modèle :", EMBEDDING_MODEL_NAME)
embeddings = get_embeddings()
print("Modèle chargé.")

Chargement du modèle : sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Modèle chargé.


## 2b. Comprendre un embedding (avant de tout indexer)

Regardons à quoi ressemble concrètement l'embedding d'un seul chunk : un vecteur de
nombres, pas du texte. C'est ce vecteur qui permettra de comparer mathématiquement
la "proximité de sens" entre deux textes.

In [3]:
sample_text = chunks[0].page_content
sample_vector = embeddings.embed_query(sample_text)

print("Texte du chunk (extrait) :", sample_text[:150], "...")
print("\nDimension du vecteur :", len(sample_vector))
print("Premières valeurs du vecteur :", sample_vector[:8])

Texte du chunk (extrait) : # Processus métier — GDS GED Services (document d'exemple pour test)

## Atelier Réception

Les dossiers papier arrivent chez GDS GED Services et sont ...

Dimension du vecteur : 384
Premières valeurs du vecteur : [-0.30553972721099854, 0.06379536539316177, -0.15237119793891907, -0.011630933731794357, -0.1169091984629631, -0.15080346167087555, 0.007715324871242046, -0.21880581974983215]


## 2c. Construction de l'index FAISS

FAISS calcule l'embedding de chaque chunk et les organise dans une structure qui permet
de retrouver rapidement, pour une question donnée, les chunks dont le vecteur est le
plus proche (similarité cosinus / distance euclidienne selon la configuration).

⚠️ Cette cellule peut prendre un peu de temps si tu as beaucoup de chunks (calcul d'un
embedding par chunk).

In [4]:
from langchain_community.vectorstores import FAISS
import time

start = time.time()
vectorstore = FAISS.from_documents(chunks, embeddings)
elapsed = time.time() - start

print(f"Index FAISS construit en {elapsed:.1f} secondes pour {len(chunks)} chunks.")

Index FAISS construit en 2.6 secondes pour 63 chunks.


## 2d. Tester une recherche de similarité (sans LLM)

C'est l'étape la plus importante à bien comprendre : avant même d'appeler un LLM, on
peut déjà vérifier que le retriever renvoie les BONS chunks pour une question donnée.

Essaie de changer `question` ci-dessous avec tes propres questions et observe quels
chunks reviennent, et avec quel score de distance (plus le score est **petit**, plus le
chunk est proche de la question — attention, ce n'est pas un pourcentage de confiance).

In [5]:
question = "Comment fonctionne l'atelier Préparation ?"

results = vectorstore.similarity_search_with_score(question, k=3)

print(f"Question : {question}\n")
for i, (doc, score) in enumerate(results):
    print(f"--- Résultat {i+1} (score de distance : {score:.4f}) ---")
    print("Source :", doc.metadata.get("source"))
    print(doc.page_content[:400])
    print()

Question : Comment fonctionne l'atelier Préparation ?

--- Résultat 1 (score de distance : 11.4612) ---
Source : C:\Users\DELL\Desktop\PROJET_PFA\rag-assistant-ged\data\docs\GDS_GED_Services (4).pdf
2.1.2 Organisation des ateliers
Chaque étape du processus est réalisée par un atelier spécialisé afin de garantir une
organisation efficace, une meilleure traçabilité des dossiers et un niveau élevé de qualité.
Cette organisation en ateliers permet de répartir les tâches de manière séquentielle, chaque
5

--- Résultat 2 (score de distance : 11.4678) ---
Source : C:\Users\DELL\Desktop\PROJET_PFA\rag-assistant-ged\data\docs\exemple_atelier_preparation.md
## Atelier Préparation

L'atelier Préparation consiste à :
1. Ouvrir chaque dossier du lot.
2. Classer les pièces à l'intérieur du dossier dans le bon ordre.
3. Vérifier la complétude du dossier (toutes les pièces attendues sont présentes).
4. Retirer les agrafes et trombones qui pourraient endommager le scanner.

Une erreur fréquente à ce st

In [6]:
# Essaie avec une deuxième question pour comparer
question_2 = "Quelle est la différence entre PEC et RD ?"

results_2 = vectorstore.similarity_search_with_score(question_2, k=3)

print(f"Question : {question_2}\n")
for i, (doc, score) in enumerate(results_2):
    print(f"--- Résultat {i+1} (score de distance : {score:.4f}) ---")
    print("Source :", doc.metadata.get("source"))
    print(doc.page_content[:400])
    print()

Question : Quelle est la différence entre PEC et RD ?

--- Résultat 1 (score de distance : 8.0549) ---
Source : C:\Users\DELL\Desktop\PROJET_PFA\rag-assistant-ged\data\docs\exemple_atelier_preparation.md
## Différence entre PEC et RD

PEC (Prise En Charge) et RD (Refus de Dossier) sont deux types de décisions
possibles à l'issue du traitement d'un dossier. La PEC signifie que le dossier est
accepté et pris en charge par l'organisme concerné. Le RD signifie que le dossier
est refusé, généralement parce qu'il est incomplet ou ne respecte pas les critères
requis.

## Atelier Contrôle Qualité

Le cont

--- Résultat 2 (score de distance : 16.8798) ---
Source : C:\Users\DELL\Desktop\PROJET_PFA\rag-assistant-ged\data\docs\GDS_GED_Services (4).pdf
CHAPITRE 2. CADRE DU SERVICE ET PROCESSUS DOCUMENTAIRES
Réception des dossiers
Création et enregistrement des lots
Préparation des dossiers
Numérisation
Contrôle qualité
Reclassement
Archivage
Figure 2.2– Workflow global du traitement documentaire.
2

**Ce qu'il faut vérifier ici :** les chunks renvoyés en premier (score le plus bas)
doivent contenir effectivement la réponse à la question. Si ce n'est pas le cas
(les résultats semblent hors sujet), c'est souvent un signe qu'il faut :
- ajuster la taille des chunks (étape 1c du notebook précédent),
- ou reformuler/enrichir le contenu des documents sources.

Si les résultats te semblent pertinents, tu peux passer à la suite.

## 2e. Sauvegarde de l'index sur disque

On sauvegarde l'index dans `faiss_index/` pour que l'API et le notebook 3 puissent le
recharger sans tout recalculer à chaque fois.

In [7]:
from build_index import INDEX_DIR

INDEX_DIR.mkdir(parents=True, exist_ok=True)
vectorstore.save_local(str(INDEX_DIR))
print("Index sauvegardé dans :", INDEX_DIR)

Index sauvegardé dans : C:\Users\DELL\Desktop\PROJET_PFA\rag-assistant-ged\faiss_index


## Conclusion et prochaine étape

Tu as maintenant :
- un modèle d'embeddings chargé et testé,
- un index FAISS construit et sauvegardé,
- la confirmation (via les recherches de similarité) que le retriever renvoie des
  chunks pertinents pour tes questions.

**Note :** la prochaine fois que tu modifies `data/docs/`, il suffira de relancer
```
python src/build_index.py
```
(équivalent en script de ce que ce notebook vient de faire), sans repasser par le notebook.

Prochaine étape : **`03_rag_chain.ipynb`** — brancher Ollama/Llama 3 pour générer une
vraie réponse en langage naturel à partir de ces chunks.